In [3]:
!pip install transformers
!pip install requests

In [4]:
import requests
import json
import pandas as pd

# Step 1: Search for a school and get its ID
def get_school_id(school_name):
    url = "https://www.ratemyprofessors.com/graphql"
    
    headers = {
        "Authorization": "Basic dGVzdDp0ZXN0",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://www.ratemyprofessors.com/",
    }
    
    query = """
    query SchoolSearchQuery($query: SchoolSearchQuery!) {
        newSearch {
            schools(query: $query) {
                edges {
                    node {
                        id
                        name
                        city
                        state
                    }
                }
            }
        }
    }
    """
    
    payload = {"query": query, "variables": {"query": {"text": school_name}}}
    response = requests.post(url, json=payload, headers=headers)
    
    print("Status:", response.status_code)  # add this temporarily
    data = response.json()
    
    if "errors" in data:
        print("GraphQL errors:", data["errors"])
        return []
    
    schools = data["data"]["newSearch"]["schools"]["edges"]
    for school in schools:
        print(school["node"])
    
    return schools


results = get_school_id("Purdue University West Lafayette")

Status: 200
{'city': 'West Lafayette', 'id': 'U2Nob29sLTc4Mw==', 'name': 'Purdue University - West Lafayette', 'state': 'IN'}
{'city': 'West Lafayette', 'id': 'U2Nob29sLTE3NTk5', 'name': 'Purdue University Global', 'state': 'IN'}
{'city': 'Hammond', 'id': 'U2Nob29sLTc4NA==', 'name': 'Purdue University Northwest', 'state': 'IN'}
{'city': 'Kokomo', 'id': 'U2Nob29sLTEzMzMw', 'name': 'Purdue University - Kokomo', 'state': 'IN'}
{'city': '', 'id': 'U2Nob29sLTc4NQ==', 'name': 'Purdue University Northwest', 'state': ''}
{'city': 'Columbus', 'id': 'U2Nob29sLTQzOA==', 'name': 'Indiana University - Purdue University', 'state': 'IN'}
{'city': 'Glendale', 'id': 'U2Nob29sLTEzNDgz', 'name': 'Arizona State University - West', 'state': 'AZ'}
{'city': 'Fort Wayne', 'id': 'U2Nob29sLTg0NTg=', 'name': 'Purdue University - Fort Wayne', 'state': 'IN'}


In [6]:
# Step 2: Get professors at that school
def get_professors(school_id, num_professors=1000):
    url = "https://www.ratemyprofessors.com/graphql"
    
    headers = {
        "Authorization": "Basic dGVzdDp0ZXN0",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://www.ratemyprofessors.com/",
    }
    
    query = """
    query TeacherSearchQuery($schoolID: ID!, $count: Int!) {
        newSearch {
            teachers(query: {schoolID: $schoolID}, first: $count) {
                edges {
                    node {
                        id
                        firstName
                        lastName
                        department
                        avgRating
                        avgDifficulty
                        numRatings
                    }
                }
            }
        }
    }
    """
    
    payload = {
        "query": query,
        "variables": {"schoolID": school_id, "count": num_professors}
    }
    
    response = requests.post(url, json=payload, headers=headers)
    data = response.json()

    if "errors" in data:
        print("GraphQL errors:", data["errors"])
        return pd.DataFrame()
    
    professors = []
    for edge in data["data"]["newSearch"]["teachers"]["edges"]:
        professors.append(edge["node"])
    
    return pd.DataFrame(professors)


# Use the school ID you got from Step 1
df_profs = get_professors("U2Nob29sLTc4Mw==", num_professors=200)
print(df_profs.head())

   avgDifficulty  avgRating         department firstName  \
0            3.2        2.7  Computer Graphics    Travis   
1            4.1        2.0     Health Science      Lisa   
2            3.4        3.5       Biochemistry      Orla   
3            2.0        4.7            English  Jonathan   
4            3.0        3.5   Computer Science   Michael   

                     id   lastName  numRatings  
0  VGVhY2hlci0yNzgzMTQy     Fuerst          76  
1  VGVhY2hlci0xODg1NTgz   Hilliard          75  
2  VGVhY2hlci0xOTMyNTQ5       Hart          27  
3  VGVhY2hlci0yMTc5MzMy      Moore          25  
4  VGVhY2hlci0zMDU4NjAz  Borkowski          22  


In [7]:
# Step 3: Get reviews for a specific professor
def get_reviews(professor_id, num_reviews=20):
    url = "https://www.ratemyprofessors.com/graphql"
    
    headers = {
        "Authorization": "Basic dGVzdDp0ZXN0",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://www.ratemyprofessors.com/",
    }
    
    query = """
    query RatingsListQuery($id: ID!, $count: Int!) {
        node(id: $id) {
            ... on Teacher {
                firstName
                lastName
                ratings(first: $count) {
                    edges {
                        node {
                            comment
                            qualityRating
                            difficultyRatingRounded
                            date
                            class
                            thumbsUpTotal
                            thumbsDownTotal
                        }
                    }
                }
            }
        }
    }
    """
    
    payload = {
        "query": query,
        "variables": {"id": professor_id, "count": num_reviews}
    }
    
    response = requests.post(url, json=payload, headers=headers)
    data = response.json()

    if "errors" in data:
        print("GraphQL errors:", data["errors"])
        return pd.DataFrame()

    node = data["data"]["node"]
    if node is None:
        print(f"No data found for professor ID: {professor_id}")
        return pd.DataFrame()

    reviews = []
    for edge in node["ratings"]["edges"]:
        r = edge["node"]
        r["professor"] = f"{node['firstName']} {node['lastName']}"
        reviews.append(r)
    
    return pd.DataFrame(reviews)


# Use any professor ID from your df_profs dataframe
df_reviews = get_reviews(df_profs["id"][0])
print(df_reviews.head())

     class                                            comment  \
0   CGT163  I went into this class with zero CAD backgroun...   
1  MFET163  As difficult as he seems to be, He is transpar...   
2  MFET163  Overall, I found this course very beneficial, ...   
3  MFET163  This class has a bad rep, mostly because it wa...   
4  MFET163  The content isn't too bad but the class is ver...   

                            date  difficultyRatingRounded  qualityRating  \
0  2026-04-26 03:14:56 +0000 UTC                        3              5   
1  2026-04-21 16:21:12 +0000 UTC                        3              4   
2  2026-04-10 00:03:30 +0000 UTC                        2              5   
3  2026-03-12 19:31:10 +0000 UTC                        4              4   
4  2026-02-24 01:03:09 +0000 UTC                        3              1   

   thumbsDownTotal  thumbsUpTotal      professor  
0                0              0  Travis Fuerst  
1                0              0  Travis Fuerst  

In [8]:
from tqdm import tqdm
all_reviews = []

for _, prof in tqdm(df_profs.iterrows(), total=len(df_profs)):
    # skip professors with very few reviews
    if prof["numRatings"] < 5:
        continue
    
    try:
        reviews = get_reviews(prof["id"], num_reviews=20)
        reviews["department"] = prof["department"]
        reviews["avgRating"] = prof["avgRating"]
        all_reviews.append(reviews)
    except Exception as e:
        print(f"Failed for {prof['firstName']}: {e}")
        continue

df_all = pd.concat(all_reviews, ignore_index=True)
df_all.to_csv("rmp_reviews.csv", index=False)
print(f"Collected {len(df_all)} reviews across {len(df_profs)} professors")

100%|██████████| 200/200 [02:48<00:00,  1.19it/s]

Collected 2534 reviews across 200 professors


In [10]:
import pandas as pd

df = pd.read_csv("rmp_reviews.csv")
print(df.shape)
print(df.head())

df = df.dropna(subset=["comment"])
df = df[df["comment"].str.len() > 10]
print(f"Clean reviews: {len(df)}")

(2534, 10)
     class                                            comment  \
0   CGT163  I went into this class with zero CAD backgroun...   
1  MFET163  As difficult as he seems to be, He is transpar...   
2  MFET163  Overall, I found this course very beneficial, ...   
3  MFET163  This class has a bad rep, mostly because it wa...   
4  MFET163  The content isn't too bad but the class is ver...   

                            date  difficultyRatingRounded  qualityRating  \
0  2026-04-26 03:14:56 +0000 UTC                        3              5   
1  2026-04-21 16:21:12 +0000 UTC                        3              4   
2  2026-04-10 00:03:30 +0000 UTC                        2              5   
3  2026-03-12 19:31:10 +0000 UTC                        4              4   
4  2026-02-24 01:03:09 +0000 UTC                        3              1   

   thumbsDownTotal  thumbsUpTotal      professor         department  avgRating  
0                0              0  Travis Fuerst  Computer G

In [11]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

c:\Users\saanv\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 2704.92it/s]


In [13]:
aspects = [
    "harsh grading",
    "great teaching",
    "poor teaching",
    "engaging lectures",
    "boring lectures",
    "heavy workload",
    "approachable and kind",
    "unavailable or unhelpful"
]

In [14]:
sample = df["comment"].iloc[0]
print("Review:", sample)

result = classifier(sample, aspects, multi_label=True)

for label, score in zip(result["labels"], result["scores"]):
    bar = "█" * int(score * 20)
    print(f"{label:<25} {bar} {score:.2%}")

Review: I went into this class with zero CAD background, so I was kinda freaked out about this class, but it wasn't bad at all.  Learned a lot of NX and Teamcenter stuff and it actually helped me get an internship.  Textbook is expensive, but it did help me understand CAD with the practice exercises.
great teaching            ████████████████ 84.19%
approachable and kind     ███████████████ 75.48%
engaging lectures         █████ 27.19%
heavy workload            █ 9.30%
poor teaching              3.31%
harsh grading              1.16%
unavailable or unhelpful   0.06%
boring lectures            0.03%


In [15]:
from tqdm import tqdm
tqdm.pandas()

def classify_review(comment):
    try:
        result = classifier(comment, aspects, multi_label=True)
        # return a dict of {aspect: score}
        return dict(zip(result["labels"], result["scores"]))
    except Exception as e:
        return {}

# This will take several minutes
df["aspect_scores"] = df["comment"].progress_apply(classify_review)

100%|██████████| 2526/2526 [17:22:00<00:00, 24.75s/it]      


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [7]:
from transformers import pipeline

"""
classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None  # returns scores for ALL emotions, not just the top one
)
"""

'\nclassifier = pipeline(\n    "text-classification",\n    model="j-hartmann/emotion-english-distilroberta-base",\n    top_k=None  # returns scores for ALL emotions, not just the top one\n)\n'

In [33]:

classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      hypothesis_template="This student writing this review found the class to be {}.")

review = "She grades really harshly and curves nothing, but her lectures are incredibly clear."

candidate_labels = ["harsh grading", "poor teaching", "great teaching", 
                    "boring", "fun and lively",
                      "heavy workload", "approachable", "confusing explanations"]
results = classifier(review, candidate_labels)
print(results)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 6760.22it/s]


{'sequence': 'She grades really harshly and curves nothing, but her lectures are incredibly clear.', 'labels': ['harsh grading', 'great teaching', 'approachable', 'heavy workload', 'poor teaching', 'confusing explanations', 'boring', 'fun and lively'], 'scores': [0.8378264307975769, 0.07927465438842773, 0.04217296838760376, 0.0203534085303545, 0.015899280086159706, 0.0025271440390497446, 0.0010775469709187746, 0.0008685376378707588]}


In [36]:
labels = results['labels']
scores = results['scores']
pairs = sorted(zip(labels, scores), key=lambda x: x[1], reverse=True)

for label, score in pairs:
    bar = "█" * int(score * 20)     
    print(f"{label:<15} {bar} {score:.2%}")  


harsh grading   ████████████████ 83.78%
great teaching  █ 7.93%
approachable     4.22%
heavy workload   2.04%
poor teaching    1.59%
confusing explanations  0.25%
boring           0.11%
fun and lively   0.09%


In [37]:
texts = [
    "So mean for no reason ughhh!",
    "Can't teach anything properly I'm always so confused."
    "One of the best professors there! Makes the class super fun!"
]

for text in texts:
    result = classifier(text, candidate_labels)
    top_idx = result['scores'].index(max(result['scores']))
    top_label = result['labels'][top_idx]
    top_score = result['scores'][top_idx]
    print(f"'{text[:40]}...' → {top_label} ({top_score:.2%})")

'So mean for no reason ughhh!...' → poor teaching (53.03%)
'Can't teach anything properly I'm always...' → fun and lively (53.94%)
